# 🔴 LLM Red Teaming — Notebook 1: Adversarial NLP Attacks

## Overview

This notebook explores how **adversarial text perturbations** can degrade the performance of large language models on a standard NLP classification task.

Adversarial attacks on NLP models work by making small, deliberate modifications to input text — changes that are often imperceptible or trivial to a human reader, yet can cause a model to misclassify or behave unexpectedly. Understanding these attacks is a core part of AI red teaming: measuring where a model's robustness breaks down before an adversary finds it first.

---

### What this notebook covers

| Step | Description |
|---|---|
| **0. Setup** | Import all modules from the `llm_red_teaming` toolkit |
| **1. Dataset** | Load the SST-2 sentiment benchmark |
| **2. Instantiate** | Initialise all 7 attack classes and the GPT-4o target |
| **3. Sanity check** | Visually inspect what each attack does to real sentences |
| **4. Evaluation** | Run all attacks end-to-end, measuring accuracy on original vs. perturbed inputs |
| **5. Results** | Summarise accuracy drop across all attacks in a comparison table |
| **6. Visualisation** | Bar chart of accuracy drop per attack |
| **7. Save** | Persist results to CSV for further analysis |
| **8. Observations** | Interpret findings and note key insights |

---

### Attack taxonomy

The 7 attacks span 4 levels of linguistic abstraction — from raw character edits to meaning-preserving paraphrases:

| Level | Attack | Core Method | Meaning Preserved |
|---|---|---|---|
| **Character** | TextBugger | Substitute one character near word start | ❌ |
| **Character** | DeepWordBug | Insert / delete / swap one character | ❌ |
| **Word** | TextFooler | Replace word with WordNet synonym | ~✅ |
| **Word** | BERTAttack | Replace word with BERT fill-mask prediction, filtered by cosine similarity | ✅ |
| **Sentence** | CheckList | Append random alphanumeric noise token | ❌ |
| **Sentence** | StressTest | Append tautological phrase ("and true is true") | ❌ |
| **Semantic** | SemanticAttack | POS-aware WordNet synonym swap (noun/verb/adj/adv) | ✅ |

---

> **Design note:** All attack logic, target connectors, and metrics live in the `attacks/`, `targets/`, and `evaluate/` modules respectively. This notebook is intentionally **code-light** — it imports, runs, and interprets; it does not implement.

---
## Step 0 · Environment Setup

### 0a — Install dependencies

We install all required packages using `sys.executable` — this ensures `pip` targets the **exact Python interpreter that this Jupyter kernel is running**, rather than a different system-level Python.

> ⚠️ **Why not `!pip install`?**  
> `!pip install <pkg>` runs pip against your shell's default Python, which may differ from the kernel's environment (e.g. a conda env or a venv). Using `!{sys.executable} -m pip install` guarantees packages land in the right place.  
> If you see `ModuleNotFoundError` after a seemingly successful `!pip install`, this mismatch is almost always the cause.

Run this cell once before proceeding. It is safe to re-run — already-installed packages are skipped automatically.

In [ ]:
import sys

# ── Install into THIS kernel's Python — not the system Python ─────────────────
# -q suppresses verbose output; remove it if you want to see what's happening
!{sys.executable} -m pip install -q \
    nltk \
    transformers \
    sentence-transformers \
    openai \
    python-dotenv \
    pandas \
    matplotlib \
    seaborn \
    tqdm

print(f"✅ Packages installed into kernel: {sys.executable}")

### 0b — Imports and path setup

We add the repo root to `sys.path` so all local modules (`attacks/`, `targets/`, `evaluate/`) resolve correctly, then load API credentials from a `.env` file.

**Modules imported:**
- `attacks.*` — all 7 adversarial attack classes
- `targets.AzureOpenAITarget` — pluggable wrapper around the Azure OpenAI chat completions API
- `evaluate.metrics` — standardised accuracy drop and reporting functions

> **Prerequisite:** Copy `.env.example` → `.env` and fill in `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, and `AZURE_OPENAI_MODEL` before running.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv('../.env')

# ── Attack imports ────────────────────────────────────────────────────────────
from attacks.character import TextBugger, DeepWordBug
from attacks.word      import TextFooler, BERTAttack
from attacks.sentence  import CheckList, StressTest
from attacks.semantic  import SemanticAttack

# ── Target & evaluate imports ─────────────────────────────────────────────────
from targets.azure_openai import AzureOpenAITarget
from evaluate.metrics     import accuracy_drop, adversarial_report

print('✅ All modules loaded')
print(f'   Python kernel : {sys.executable}')

✅ All modules loaded
   Python kernel : /Users/minwu/miniconda3/envs/adv_env/bin/python


---
## Step 1 · Load the SST-2 Dataset

### What is SST-2?

[SST-2 (Stanford Sentiment Treebank 2)](https://huggingface.co/datasets/stanfordnlp/sst2) is a binary sentiment classification dataset derived from movie reviews. It is part of the [GLUE benchmark](https://gluebenchmark.com/) and is one of the most widely used datasets in NLP research.

Each example contains:
- **`sentence`** — a short movie review snippet
- **`label`** — `1` = Positive sentiment, `0` = Negative sentiment

### Why SST-2 for adversarial evaluation?

SST-2 is an ideal adversarial testbed for two reasons:

1. **Sensitivity to small changes** — Sentiment can flip with a single word substitution (e.g. `"good"` → `"mediocre"`), making it a strong signal for detecting adversarial impact.
2. **Lexical diversity** — The dataset contains varied vocabulary, short and long sentences, and domain-specific film language — all properties that stress-test different attack types differently.

We use the **development split** (`dev.tsv`, 872 rows) to avoid any training data contamination.

### Data loading strategy

The cell below uses a **two-step fallback**:

1. **Repo copy first** — `data/sst2/dev.tsv` is bundled in the repository so the notebook works immediately for all users, with no downloads or local paths needed.
2. **HuggingFace fallback** — If the repo file is missing (e.g. a custom fork or manual deletion), the dataset is downloaded automatically from `stanfordnlp/sst2` on HuggingFace and saved locally for next time.

> **No configuration needed** — just run the cell.

In [2]:
import os, pandas as pd

# ── Repo-bundled path (works for all users, no config needed) ──────────────────
REPO_DEV  = os.path.join(os.path.abspath('..'), 'data', 'sst2', 'dev.tsv')
REPO_TRAIN = os.path.join(os.path.abspath('..'), 'data', 'sst2', 'train_sample.tsv')

def _download_sst2_hf(save_dir: str) -> None:
    """Fallback: download SST-2 from HuggingFace and save locally."""
    print("  Repo data not found — downloading from HuggingFace (stanfordnlp/sst2)...")
    from datasets import load_dataset
    os.makedirs(save_dir, exist_ok=True)
    ds = load_dataset('stanfordnlp/sst2')
    ds['validation'].to_pandas()[['sentence','label']]\
        .to_csv(os.path.join(save_dir, 'dev.tsv'), sep='\t', index=False)
    ds['train'].to_pandas()[['sentence','label']]\
        .sample(2000, random_state=42)\
        .to_csv(os.path.join(save_dir, 'train_sample.tsv'), sep='\t', index=False)
    print("  Saved to data/sst2/ for future use.")

# ── Load with fallback ─────────────────────────────────────────────────────────
if os.path.exists(REPO_DEV):
    print("✅ Loading bundled repo data...")
    dev_df   = pd.read_csv(REPO_DEV,   sep='\t')
    train_df = pd.read_csv(REPO_TRAIN, sep='\t')
else:
    _download_sst2_hf(os.path.dirname(REPO_DEV))
    dev_df   = pd.read_csv(REPO_DEV,   sep='\t')
    train_df = pd.read_csv(REPO_TRAIN, sep='\t')

print(f"\n  dev set          : {len(dev_df):,} rows")
print(f"  train sample     : {len(train_df):,} rows")
print(f"\n  Label distribution (dev):")
print(dev_df['label'].value_counts().rename({1: 'positive', 0: 'negative'}).to_string())
dev_df.head()

✅ Loading bundled repo data...

  dev set          : 872 rows
  train sample     : 2,000 rows

  Label distribution (dev):
label
positive    444
negative    428


,sentence,label
0,it 's a charming and often affecting journey .,1
1,unflinchingly bleak and desperate,0
2,allows us to hope that nolan is poised to emba...,1
3,"the acting , costumes , music , cinematography...",1
4,"it 's slow -- very , very slow .",0


---
## Step 2 · Instantiate Attacks and Target Model

### Attack instantiation

Each attack is a self-contained class implementing a common interface: `attack(text: str) -> str` and `attack_batch(texts: list[str]) -> list[str]`. All stochastic attacks accept a `seed` argument for reproducibility.

| Attack | Key Parameters | Notes |
|---|---|---|
| `TextBugger` | `seed` | Deterministic single-char substitution at position 3 |
| `DeepWordBug` | `seed` | Randomly picks insert / delete / swap on each call |
| `TextFooler` | `seed`, `min_word_len`, `top_k` | Skips stopwords; samples from top-3 synonyms |
| `BERTAttack` | `bert_model_path`, `sim_threshold` | Requires local BERT + SentenceTransformer; slowest attack |
| `CheckList` | `seed`, `token_length` | Appends 10-char alphanumeric noise |
| `StressTest` | `seed` | Picks from a set of tautological suffixes |
| `SemanticAttack` | `min_word_len` | No randomness — deterministic WordNet lookup |

### Target model

The **victim model** is `GPT-4o` served via Azure OpenAI. It receives each (original and attacked) sentence and is asked to classify sentiment as `"positive"` or `"negative"`. This replicates a realistic deployment scenario where the model is being used as an NLP classifier via a prompt.

> **Note:** `BERTAttack` loads two local models at instantiation time (BERT fill-mask + SentenceTransformer). This may take 15–30 seconds on first run.

In [3]:
# Model target — reads credentials from .env
target = AzureOpenAITarget()
print(target)

# All 7 attacks — seeded for reproducibility
attacks = {
    'TextBugger':     TextBugger(seed=42),
    'DeepWordBug':    DeepWordBug(seed=42),
    'TextFooler':     TextFooler(seed=42),
    'BERTAttack':     BERTAttack(),        # loads BERT & SentenceTransformer on init
    'CheckList':      CheckList(seed=42),
    'StressTest':     StressTest(seed=42),
    'SemanticAttack': SemanticAttack(),
}
print(f'\n{len(attacks)} attacks ready:')
for name, atk in attacks.items():
    print(f'  {name:18s}  {atk}')

OpenAICompatibleTarget(provider='openai', model='gpt-5-4-20260305-gs', endpoint='https://atlas.protiviti.com/Experiment20240821')


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


7 attacks ready:
  TextBugger          TextBugger(level='character', method='char_substitution')
  DeepWordBug         DeepWordBug(level='character', method='insert|delete|swap')
  TextFooler          TextFooler(level='word', top_k=3)
  BERTAttack          BERTAttack(level='word', top_k=5, sim_threshold=0.85)
  CheckList           CheckList(level='sentence', token_length=10)
  StressTest          StressTest(level='sentence', method='tautology_append')
  SemanticAttack      SemanticAttack(level='semantic', method='pos_aware_synonym')


---
## Step 3 · Sanity Check — Visual Inspection of Attack Outputs

### Why this step matters

Before running a full evaluation, it is essential to visually inspect what each attack actually does to real sentences. This serves two purposes:

1. **Validation** — Confirms the attack is working as intended (e.g. TextBugger really is substituting a character, not a whole word).
2. **Intuition building** — Seeing the outputs side-by-side helps calibrate expectations for which attacks are more or less aggressive, and which might plausibly fool a human or a model.

### What to look for

- **Character-level attacks** (TextBugger, DeepWordBug): look for single-character changes, likely mid-word. The sentence should still be mostly readable.
- **Word-level attacks** (TextFooler, BERTAttack): one word should change to a synonym or a contextually similar word. The sentence should remain grammatical.
- **Sentence-level attacks** (CheckList, StressTest): the original sentence is unchanged; something is appended at the end.
- **Semantic attack** (SemanticAttack): one content word (noun, verb, adjective, adverb) should be replaced with a meaning-preserving synonym.

In [4]:
sample = dev_df.sample(5, random_state=42)['sentence'].tolist()

for name, atk in attacks.items():
    print(f'\n── {name} ──')
    for sent in sample:
        attacked = atk.attack(sent)
        changed  = '⚠️ ' if attacked != sent else '✅ '
        print(f'  original : {sent}')
        print(f'  attacked : {changed}{attacked}')
        print()


── TextBugger ──
  original : it confirms fincher 's status as a film maker who artfully bends technical know-how to the service of psychological insight . 
  attacked : ⚠️ it Oonfirms fincher 's status as a film maker who artfully bends technical know-how to the service of psychological insight . 

  original : too much of it feels unfocused and underdeveloped . 
  attacked : ⚠️ toohmuch of it feels unfocused and underdeveloped . 

  original : a great ensemble cast ca n't lift this heartfelt enterprise out of the familiar . 
  attacked : ⚠️ a gbeat ensemble cast ca n't lift this heartfelt enterprise out of the familiar . 

  original : prurient playthings aside , there 's little to love about this english trifle . 
  attacked : ⚠️ pruVient playthings aside , there 's little to love about this english trifle . 

  original : it moves quickly , adroitly , and without fuss ; it does n't give you time to reflect on the inanity -- and the cold war datedness -- of its premise . 
  attacke

---
## Step 4 · Full Evaluation Loop

### Evaluation design

For each attack, we run the following loop over `NUM_SAMPLES` sentences from the SST-2 dev set:

```
for each sentence:
    1. Send original sentence → GPT-4o → record predicted label
    2. Apply attack to sentence → perturbed sentence
    3. Send perturbed sentence → GPT-4o → record predicted label
    4. Compare both predictions against the ground-truth label
```

This yields two accuracy figures per attack — **original accuracy** and **attacked accuracy** — from which we compute the **accuracy drop**:

> `accuracy_drop = original_accuracy − attacked_accuracy`

A higher drop means the attack was more effective at degrading model performance.

### API call budget

| Parameter | Value | API calls |
|---|---|---|
| `NUM_SAMPLES` | 50 | 2 calls × 50 samples × 7 attacks = **700 calls** |
| `NUM_SAMPLES` | 100 | 2 calls × 100 samples × 7 attacks = **1,400 calls** |

A `SLEEP` delay between calls is enforced to stay within Azure OpenAI rate limits. Reduce `NUM_SAMPLES` for a quick test run.

> **Runtime estimate:** At `SLEEP=1.2s` and 7 attacks × 50 samples × 2 calls, expect approximately **14 minutes** of wall-clock time.

In [5]:
import time

NUM_SAMPLES = 50   # ← increase to 100 for a full evaluation
SLEEP = 1.2        # seconds between API calls to avoid rate limiting

eval_df = dev_df.head(NUM_SAMPLES).reset_index(drop=True)
results_by_attack: dict[str, dict] = {}

for attack_name, atk in attacks.items():
    print(f'\n{"="*50}')
    print(f'  Running {attack_name} on {NUM_SAMPLES} samples...')
    print(f'{"="*50}')

    orig_correct = 0
    atk_correct  = 0
    orig_failed  = 0
    atk_failed   = 0

    for _, row in eval_df.iterrows():
        text  = row['sentence']
        label = 'positive' if row['label'] == 1 else 'negative'

        # ── Original prediction (with graceful failure handling) ───────────
        try:
            orig_pred = target.get_sentiment(text)
            if orig_pred == label:
                orig_correct += 1
        except Exception as e:
            orig_pred = 'error'
            orig_failed += 1
            print(f'    ❌ original call failed after retries: {type(e).__name__}: {str(e)[:80]}')

        # ── Attacked prediction (with graceful failure handling) ───────────
        try:
            attacked_text = atk.attack(text)
            atk_pred = target.get_sentiment(attacked_text)
            if atk_pred == label:
                atk_correct += 1
        except Exception as e:
            atk_pred = 'error'
            atk_failed += 1
            print(f'    ❌ attacked call failed after retries: {type(e).__name__}: {str(e)[:80]}')

        time.sleep(SLEEP)

    # Accuracy computed over successfully returned predictions
    effective_n = NUM_SAMPLES - max(orig_failed, atk_failed)
    metrics = accuracy_drop(orig_correct, atk_correct, max(effective_n, 1))
    metrics['orig_failed'] = orig_failed
    metrics['atk_failed']  = atk_failed
    results_by_attack[attack_name] = metrics

    print(f'  → original_acc={metrics["original_acc"]:.2%}  '
          f'attacked_acc={metrics["attacked_acc"]:.2%}  '
          f'acc_drop={metrics["acc_drop"]:.2%}  '
          f'(failures: orig={orig_failed}, atk={atk_failed})')

print('\n✅ Evaluation complete')


  Running TextBugger on 50 samples...


InternalServerError: Error code: 500 - {'statusCode': 500, 'message': 'Internal server error', 'activityId': '6ed3ac87-67c9-4766-afed-75fd960820dc'}

---
## Step 5 · Results Summary Table

The table below ranks all attacks by accuracy drop (highest impact first).

**How to read the table:**

| Column | Meaning |
|---|---|
| `original_acc` | GPT-4o accuracy on unmodified SST-2 sentences |
| `attacked_acc` | GPT-4o accuracy on perturbed sentences |
| `acc_drop` | Difference — the core measure of attack effectiveness |

The red gradient on `acc_drop` makes the most impactful attacks immediately visible.

> **Interpretation guide:** A drop of 0% means the attack had no effect. A drop of 10%+ on a model as capable as GPT-4o is considered meaningful. Drops exceeding 20% would indicate a serious robustness vulnerability.

In [ ]:
report = adversarial_report(results_by_attack)
report.style\
      .format({'original_acc': '{:.2%}', 'attacked_acc': '{:.2%}', 'acc_drop': '{:.2%}'})\
      .background_gradient(subset=['acc_drop'], cmap='Reds')\
      .set_caption('Adversarial Attack Evaluation — SST-2 Dev Set / GPT-4o')

---
## Step 6 · Visualisation — Accuracy Drop by Attack

The bar chart below provides a quick visual ranking of attack effectiveness.

- **Taller bars** indicate more effective attacks — i.e., larger degradation of model performance after perturbation.
- The chart is sorted by `acc_drop` (descending) to make the ranking immediately clear.
- The plot is saved to `results/01_accuracy_drop.png` for inclusion in reports or presentations.

Note that a high accuracy drop does not necessarily mean the attack is *stealthy* or *realistic*. For example, `StressTest` appends nonsensical text that a human would immediately spot, whereas `BERTAttack` produces fluent, natural-sounding perturbations. Both dimensions — **impact** and **stealth** — matter in a full red-team assessment.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=report,
    x='attack', y='acc_drop',
    palette='Reds_r', ax=ax
)
ax.set_title('Accuracy Drop by Attack Type (SST-2 / GPT-4o)', fontsize=14, fontweight='bold')
ax.set_xlabel('Attack', fontsize=11)
ax.set_ylabel('Accuracy Drop', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/01_accuracy_drop.png', dpi=150)
plt.show()

---
## Step 7 · Save Results

Results are written to the `results/` directory (gitignored by default to avoid committing potentially sensitive evaluation data).

The CSV can be loaded in a future session to:
- Compare results across different models or dataset sizes
- Feed into a reporting pipeline
- Track changes as the toolkit evolves

In [ ]:
os.makedirs('../results', exist_ok=True)
out_path = '../results/01_adversarial_results.csv'
report.to_csv(out_path, index=False)
print(f'✅ Results saved → {os.path.abspath(out_path)}')
print(f'   {len(report)} attack rows, columns: {list(report.columns)}')

---
## Step 8 · Observations & Key Takeaways

### Per-attack analysis

| Attack | Level | Expected Impact on GPT-4o | Reasoning |
|---|---|---|---|
| **TextBugger** | Character | Low–Medium | GPT-4o is trained on noisy internet text and is highly tolerant of typos. Single-character substitutions near the start of a word rarely change the model's interpretation. |
| **DeepWordBug** | Character | Low–Medium | Insert/swap/delete operations mimic realistic typing errors. Impact is marginally higher than TextBugger for rare or domain-specific words that have no common misspelling context in training data. |
| **TextFooler** | Word | Medium | Synonym substitution can shift the **polarity** of sentiment-bearing words (e.g. `"wonderful"` → `"fantastic"` is safe; `"wonderful"` → `"strange"` is not). Top-k sampling introduces variability. |
| **BERTAttack** | Word | Low–Medium | By design, substitutions are filtered to high cosine similarity — meaning-preserving swaps. GPT-4o handles contextual paraphrases well, so impact tends to be low. |
| **CheckList** | Sentence | Low | Appended noise is syntactically disconnected. GPT-4o attends to the full sentence and typically ignores incoherent suffixes. |
| **StressTest** | Sentence | Low | Tautological phrases are logically vacuous. A strong LLM sees through the noise. Impact would be higher on smaller, less capable models. |
| **SemanticAttack** | Semantic | Low–Medium | POS-constrained synonym substitution is the most human-like attack here. It can affect adjectives and verbs that directly carry sentiment, but NLTK synonyms are not always contextually appropriate. |

---

### Broader findings

**GPT-4o is substantially more robust than smaller fine-tuned models** to all attacks tested here. This is consistent with findings in the literature: instruction-tuned LLMs developed robustness to surface-form perturbations as a side effect of large-scale pretraining on diverse, noisy corpora.

However, this does **not** mean GPT-4o is immune to adversarial inputs — it means character- and word-level perturbations are not the right attack surface. More effective attacks target the *alignment layer* (jailbreaking, covered in Notebook 2) rather than the *input representation layer*.

**Key design trade-off — impact vs. stealth:**

| Attack type | Impact on model | Detectability by human |
|---|---|---|
| Character-level | Low (on LLMs) | High (obvious typos) |
| Word synonym | Medium | Low (fluent text) |
| Sentence append | Low | High (unnatural suffix) |
| Semantic paraphrase | Low–Medium | Very low (reads naturally) |

The most dangerous adversarial inputs combine **low detectability** with **high model impact** — a combination better achieved through semantic and alignment-level attacks.

---

### Next steps

- 📓 **Notebook 2** — Jailbreaking: attacks at the alignment layer using JailbreakBench
- 🔜 **Phase 2** — Prompt injection (direct + indirect / RAG pipelines)
- 🔜 **Phase 3** — Bias & fairness testing (BBQ, WinoBias)